# Glioma IDH Classification
This notebook demonstrates the full workflow with **synthetic data**. It is an executable tutorial, not a clinical analysis. Replace the demo cell with carefully curated patient-level expression and IDH labels for a real study.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.idh_model import make_demo
from src.full_analysis import run_full_analysis
print(f'Project root: {ROOT}')

## 1. Create a reproducible demonstration cohort
The simulated labels and features verify the computational workflow but do not represent patients, measured genes, or real IDH biology.

In [ ]:
X, y, sample_ids = make_demo(seed=42)
print('Expression shape:', X.shape)
print('Class counts:')
print(y.value_counts().rename(index={0: 'IDH wild-type', 1: 'IDH mutant'}))
X.head()

## 2. Run model comparison and held-out evaluation
All learned transformations remain inside each pipeline, preventing test information from influencing feature selection or scaling.

In [ ]:
output_dir = ROOT / 'results' / 'notebook_demo'
summary = run_full_analysis(X, y, sample_ids, output_dir, top_k=75, seed=42, demo=True)
summary['selected_model'], summary['held_out_test'][summary['selected_model']]

## 3. Inspect cross-validation and gene importance

In [ ]:
import pandas as pd
cv = pd.read_csv(output_dir / 'tables' / 'cross_validation_results.csv')
importance = pd.read_csv(output_dir / 'tables' / 'permutation_importance.csv')
display(cv)
display(importance.head(15))

## 4. Review generated figures

In [ ]:
from IPython.display import Image, display
for name in ['pca_by_idh_status.png', 'model_comparison.png', 'roc_curves.png', 'precision_recall_curves.png', 'top_gene_importance.png']:
    display(Image(filename=str(output_dir / 'figures' / name)))

## Interpretation checklist
- Preserve patient-level separation when repeated specimens exist.
- Examine grade, histology, age, and batch as possible confounders.
- Treat importance as association, not a mechanistic result.
- Report cross-validation variability and held-out results separately.
- Validate on an independent cohort before making diagnostic claims.